# Smart MCQ Solver Challenge (DL & GenAI Project)

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Load the datasets

In [2]:
df1 = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df2 = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [3]:
print(df1.shape)
print(df2.shape)
print(sample.shape)

(2000, 8)
(500, 7)
(500, 2)


In [4]:
df1.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


## Import libraries

In [6]:

# Smart MCQ Solver
# DistilBERT Transformer Baseline


# Standard Library
import os
import random
import warnings

warnings.filterwarnings("ignore")


# Data Manipulation
import numpy as np
import pandas as pd


# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


# PyTorch
import torch


# Hugging Face
import transformers
import datasets

from datasets import Dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments


# Utilities
from scipy.special import softmax
from tqdm.auto import tqdm


# Display Versions
print("Environment Information")

print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"PyTorch      : {torch.__version__}")

print("\nGPU Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

Environment Information
Transformers : 5.0.0
Datasets     : 4.8.5
PyTorch      : 2.10.0+cu128

GPU Available : True
GPU : Tesla T4


## Set Configurations

In [82]:
# Configuration

class CFG:

    # Model
    # MODEL_NAME = "distilbert-base-uncased"

    # MODEL_NAME = "microsoft/deberta-v3-base"
    
    # Future notebooks:
    MODEL_NAME = "FacebookAI/roberta-base"
    # MODEL_NAME = "microsoft/deberta-v3-small"


    # Labels
    NUM_LABELS = 5

    LABEL2ID = {
        "A": 0,
        "B": 1,
        "C": 2,
        "D": 3,
        "E": 4,
    }

    ID2LABEL = {v: k for k, v in LABEL2ID.items()}


    # Tokenization
    MAX_LENGTH = 384

    
    # Training
    TRAIN_BATCH_SIZE = 16

    VALID_BATCH_SIZE = 32

    LEARNING_RATE = 1e-5

    WEIGHT_DECAY = 0.01

    EPOCHS = 8

    WARMUP_RATIO = 0.10

    
    # Misc
    VALID_SIZE = 0.20

    RANDOM_STATE = 42

    NUM_WORKERS = 2

In [57]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device: cuda
Tesla T4


In [9]:
train = df1.copy()
test = df2.copy()

In [83]:
### Reproducibility

def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(CFG.RANDOM_STATE)

In [11]:
# Missing Values

print("Train Missing Values")

display(train.isnull().sum())

print()

print("Test Missing Values")

display(test.isnull().sum())

Train Missing Values


id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64


Test Missing Values


id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

In [12]:
# Label Distribution

print(train["answer"].value_counts())

print()

print(
    train["answer"]
    .value_counts(normalize=True)
    .sort_index()
)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

answer
A    0.1845
B    0.2450
C    0.2295
D    0.1790
E    0.1620
Name: proportion, dtype: float64


In [13]:
## Dataset Summary

print("\nTrain Columns")

print(train.columns.tolist())

print("\nTest Columns")

print(test.columns.tolist())

print("\nSubmission Columns")

print(sample.columns.tolist())


Train Columns
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

Test Columns
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']

Submission Columns
['ID', 'Prediction']


## Dataset preprocessing

In [59]:
# Label Encoding

train["label"] = train["answer"].map(CFG.LABEL2ID)

print(train[["answer", "label"]].head())

  answer  label
0      B      1
1      A      0
2      C      2
3      B      1
4      A      0


In [84]:
### Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    CFG.MODEL_NAME
)

print(tokenizer)

RobertaTokenizer(name_or_path='FacebookAI/roberta-base', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
}
)


## Build Input Text

This is one of the most important steps.

Instead of aggressive preprocessing, preserve the natural language and combine the prompt with all answer choices into a single text sequence.

In [85]:
# Build Input Text
def build_input_text(row):
    sep = tokenizer.sep_token
    return (
        f"{row['prompt']} {sep} "
        f"{row['A']} {sep} "
        f"{row['B']} {sep} "
        f"{row['C']} {sep} "
        f"{row['D']} {sep} "
        f"{row['E']}"
    )

In [62]:
train["text"] = train.apply(build_input_text, axis=1)
test["text"] = test.apply(build_input_text, axis=1)

In [63]:
print(train["text"].iloc[0])

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. </s> Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. </s> Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement. </s> Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present. </s> Martin Heidegger believes that the relationship

## Train & Validation Split

In [86]:
# Train / Validation Split

train_df, valid_df = train_test_split(
    train,
    test_size=CFG.VALID_SIZE,
    random_state=CFG.RANDOM_STATE,
    stratify=train["label"],
)

print("Train :", train_df.shape)
print("Valid :", valid_df.shape)

Train : (1600, 10)
Valid : (400, 10)


## Tokenization

In [87]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [88]:
### Tokenization Function

### We tokenize the combined question + options text.


def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=CFG.MAX_LENGTH,
    )

In [89]:
#### Test the Tokenizer
sample_tokens = tokenizer(
    train_df.iloc[0]["text"]
)

print(sample_tokens.keys())

KeysView({'input_ids': [0, 45356, 5, 144, 6030, 1973, 35, 653, 16, 37605, 10793, 811, 22642, 36, 448, 7054, 26610, 31, 5, 511, 5717, 4, 1437, 2, 256, 7054, 16, 10, 9322, 14, 4529, 5, 3650, 9, 1109, 11, 5, 2621, 9, 670, 34217, 5447, 4, 85, 16, 41, 3626, 7, 5, 31098, 9, 2933, 948, 11, 1110, 9, 8926, 596, 30948, 109, 45, 2082, 7, 28616, 5, 855, 6238, 2074, 9, 17759, 4, 1437, 2, 256, 7054, 16, 10, 31098, 14, 21037, 10, 29685, 9, 27648, 18, 7208, 9, 937, 44999, 7, 1316, 13, 6373, 3611, 9, 30948, 4, 85, 16, 41, 3626, 7, 5, 31098, 9, 2933, 948, 11, 1110, 9, 8926, 596, 30948, 109, 45, 2082, 7, 28616, 5, 855, 6238, 2074, 9, 17759, 4, 1437, 2, 256, 7054, 16, 10, 31098, 14, 21037, 10, 29685, 9, 10793, 18, 488, 9, 10547, 20969, 12257, 7, 1316, 13, 6373, 3611, 9, 30948, 4, 85, 16, 41, 3626, 7, 5, 31098, 9, 2933, 948, 11, 1110, 9, 8926, 596, 30948, 109, 45, 2082, 7, 28616, 5, 855, 6238, 2074, 9, 17759, 4, 1437, 2, 256, 7054, 16, 10, 31098, 14, 21037, 10, 29685, 9, 33018, 5223, 18, 488, 7, 1316, 13, 

## Hugging Face dataset

In [90]:
#### Convert Pandas DataFrame to Hugging Face Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False,
)

valid_dataset = Dataset.from_pandas(
    valid_df[["text", "label"]],
    preserve_index=False,
)

test_dataset = Dataset.from_pandas(
    test[["text"]],
    preserve_index=False,
)

In [91]:
### Tokenize Datasets
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
)

valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True,
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [25]:
## Remove Unnecessary Columns
### Remove Raw Text
# train_dataset = train_dataset.remove_columns(
#     ["text"]
# )

# valid_dataset = valid_dataset.remove_columns(
#     ["text"]
# )

# test_dataset = test_dataset.remove_columns(
#     ["text"]
# )

In [92]:
### Rename Label Column

## Hugging Face expects the target column to be named labels.

train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

valid_dataset = valid_dataset.rename_column(
    "label",
    "labels"
)

## Set Dataset Format

In [93]:
## Set Dataset Format
train_dataset.set_format("torch")
valid_dataset.set_format("torch")
test_dataset.set_format("torch")

In [94]:
## verify dataset
print(train_dataset)

print(valid_dataset)

print(test_dataset)

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 1600
})
Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 400
})
Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 500
})


## Load microsoft/deberta-v3-small Model

In [95]:
## Load deBERTa-v3 Model
model = AutoModelForSequenceClassification.from_pretrained(
    CFG.MODEL_NAME,
    num_labels=CFG.NUM_LABELS,
    id2label=CFG.ID2LABEL,
    label2id=CFG.LABEL2ID,
)

model.to(device)

print(model.__class__.__name__)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RobertaForSequenceClassification


In [96]:
### Verify the Model
print(model.config)

RobertaConfig {
  "add_cross_attention": false,
  "architectures": [
    "RobertaForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "A",
    "1": "B",
    "2": "C",
    "3": "D",
    "4": "E"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



## Evaluation Metric (MAP@3)

In [97]:
def mapk(actual, predicted, k=3):
    """
    actual    : integer label
    predicted : ranked label indices
    """

    score = 0.0

    for i, p in enumerate(predicted[:k]):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score

In [98]:
## That will print both accuracy and MAP@3 score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = softmax(logits, axis=1)

    ranked = np.argsort(-probs, axis=1)

    scores = [
        mapk(y, pred, k=3)
        for y, pred in zip(labels, ranked)
    ]

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "map@3": np.mean(scores),
    }

## Training FacebookAI/Roberta-v3-base

Now we'll build the training pipeline.

In [99]:
## Training Arguments

## Since you're on Kaggle and short on time, I recommend these settings as a solid starting point.

training_args = TrainingArguments(

    output_dir="./results",

    # Required
    do_train=True,
    do_eval=True,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="map@3",
    greater_is_better=True,

    # Hyperparameters
    learning_rate=CFG.LEARNING_RATE,
    per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=CFG.VALID_BATCH_SIZE,

    num_train_epochs=CFG.EPOCHS,

    weight_decay=CFG.WEIGHT_DECAY,
    warmup_ratio=CFG.WARMUP_RATIO,

    # Logging
    logging_steps=20,

    # Saving
    save_total_limit=2,

    # Performance
    fp16=False,

    # Reproducibility
    seed=CFG.RANDOM_STATE,

    # Disable online logging
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [100]:
from transformers import TrainingArguments
import inspect

print(inspect.signature(TrainingArguments.__init__))

(self, output_dir: str | None = None, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: transformers.trainer_utils.IntervalStrategy | str = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, gradient_accumulation_steps: int = 1, eval_accumulation_steps: int | None = None, eval_delay: float = 0, torch_empty_cache_steps: int | None = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_ratio: float | None = None, warmup_steps: float = 0, log_level: str = 'passive', log_level_replica: str = 'warning', log_on_each_node: bool = True, logging_dir: str | None = None, logging_strategy: transfor

In [101]:
### Trainer
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

In [102]:
print(model)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [103]:
## Train model
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,3.209745,3.180380,0.285000,0.457500
2,2.878754,2.586915,0.507500,0.647083
3,1.864927,1.098754,0.905000,0.945417
4,0.482135,0.250059,1.000000,1.000000
5,0.164364,0.052619,1.000000,1.000000
6,0.036407,0.018272,1.000000,1.000000
7,0.022618,0.012101,1.000000,1.000000
8,0.018036,0.010983,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

## Evaluation

In [104]:
results = trainer.evaluate()

for key, value in results.items():
    print(f"{key:25s}: {value}")

eval_loss                : 0.24827010929584503
eval_accuracy            : 1.0
eval_map@3               : 1.0
eval_runtime             : 4.3917
eval_samples_per_second  : 91.081
eval_steps_per_second    : 1.594
epoch                    : 8.0


In [105]:
valid_predictions = trainer.predict(valid_dataset)

valid_logits = valid_predictions.predictions

valid_probs = softmax(valid_logits, axis=1)

valid_pred_labels = np.argmax(valid_probs, axis=1)

valid_true_labels = valid_df["label"].values

In [106]:
print("Validation Accuracy")
print("=" * 40)

accuracy = accuracy_score(
    valid_true_labels,
    valid_pred_labels
)

print(f"Accuracy : {accuracy:.5f}")

Validation Accuracy
Accuracy : 1.00000


In [107]:
## Classification reports
print(
    classification_report(
        valid_true_labels,
        valid_pred_labels,
        target_names=["A", "B", "C", "D", "E"]
    )
)

              precision    recall  f1-score   support

           A       1.00      1.00      1.00        74
           B       1.00      1.00      1.00        98
           C       1.00      1.00      1.00        92
           D       1.00      1.00      1.00        71
           E       1.00      1.00      1.00        65

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



## Create Submission

In [108]:
test_predictions = trainer.predict(test_dataset)

test_logits = test_predictions.predictions

test_probs = softmax(
    test_logits,
    axis=1
)

In [109]:
top3 = np.argsort(
    -test_probs,
    axis=1
)[:, :3]

In [110]:
### Convert IDs → Labels
top3_labels = []

for row in top3:

    prediction = " ".join(
        CFG.ID2LABEL[idx]
        for idx in row
    )

    top3_labels.append(prediction)

In [111]:
submission = sample.copy()

submission["Prediction"] = top3_labels

submission.head()

,ID,Prediction
0,1,A C E
1,2,B E C
2,3,B C D
3,4,E A B
4,5,C D A


In [112]:
submission["Prediction"].value_counts().head(20)

Prediction
D C B    51
C D A    49
B C E    48
A C E    39
B E C    36
D C A    36
E B C    29
A E C    27
E C B    17
A C D    16
B C D    16
C A D    14
E B A    13
C D B    13
C B D    13
C A B    12
B D C     9
E A B     8
C D E     8
E C A     7
Name: count, dtype: int64

In [113]:
submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")

submission.csv created successfully.


## Save the models

In [114]:

trainer.save_model("./distilbert_baseline")

tokenizer.save_pretrained("./distilbert_baseline")

print("Model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.


In [115]:
print(CFG.MAX_LENGTH)
print(CFG.LEARNING_RATE)
print(CFG.TRAIN_BATCH_SIZE)
print(CFG.VALID_BATCH_SIZE)
print(CFG.EPOCHS)
print(CFG.WARMUP_RATIO)
print(CFG.WEIGHT_DECAY)

384
1e-05
16
32
8
0.1
0.01


In [116]:
print(train_dataset[0])
print("\n", model.classifier)

{'text': "Select the most accurate option: What is Modified Newtonian Dynamics (MOND)? from the following choices. </s> MOND is a principle that explains the behavior of light in the presence of strong gravitational fields. It is an alternative to the hypothesis of dark matter in terms of explaining why galaxies do not appear to obey the currently understood laws of physics. </s> MOND is a hypothesis that proposes a modification of Einstein's framework of general relativity to account for observed properties of galaxies. It is an alternative to the hypothesis of dark matter in terms of explaining why galaxies do not appear to obey the currently understood laws of physics. </s> MOND is a hypothesis that proposes a modification of Newton's law of universal gravitation to account for observed properties of galaxies. It is an alternative to the hypothesis of dark matter in terms of explaining why galaxies do not appear to obey the currently understood laws of physics. </s> MOND is a hypoth